In [1]:
import csv
import time
import numpy as np
import scipy.sparse as sp

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim

Load the Graph data (adjacency matrix, features, and edge features)

In [2]:
def load_data():
    """
    Function that loads graphs
    """
    graph_indicator = np.loadtxt("graph_indicator.txt", dtype=np.int64)
    print(f"Graph indicator: {graph_indicator}")
    _,graph_size = np.unique(graph_indicator, return_counts=True)

    
    edges = np.loadtxt("edgelist.txt", dtype=np.int64, delimiter=",")
    
    #Make the graph undirected
    edges_inv = np.vstack((edges[:,1], edges[:,0]))
    edges = np.vstack((edges, edges_inv.T))

    #Sort the edges
    s = edges[:,0]*graph_indicator.size + edges[:,1]
    idx_sort = np.argsort(s)
    edges = edges[idx_sort,:]

    #Remove duplicate edges
    edges,idx_unique =  np.unique(edges, axis=0, return_index=True)

    #Create sparse adjacency matrix
    A = sp.csr_matrix((np.ones(edges.shape[0]), (edges[:,0], edges[:,1])), shape=(graph_indicator.size, graph_indicator.size))

    x = np.loadtxt("node_attributes.txt", delimiter=",")
    edge_attr = np.loadtxt("edge_attributes.txt", delimiter=",")

    #Duplicate and align edge attributes
    edge_attr = np.vstack((edge_attr,edge_attr))
    edge_attr = edge_attr[idx_sort,:]
    edge_attr = edge_attr[idx_unique,:]

    #initialize lists and indices
    adj = []
    features = []
    edge_features = []
    idx_n = 0
    idx_m = 0

    #Split data per graph
    for i in range(graph_size.size):
        adj.append(A[idx_n:idx_n+graph_size[i],idx_n:idx_n+graph_size[i]])
        edge_features.append(edge_attr[idx_m:idx_m+adj[i].nnz,:])
        features.append(x[idx_n:idx_n+graph_size[i],:])
        idx_n += graph_size[i]
        idx_m += adj[i].nnz

    return adj, features, edge_features


adj, features, edge_features = load_data()

Graph indicator: [   0    0    0 ... 6110 6110 6110]


Normalize the adcacency matrices and get the training data (the data set contain some test set proteins without labels, which will be filtered out)

In [3]:
# Normalize adjacency matrices

def normalize_adjacency(A):
    """
    Function that normalizes an adjacency matrix
    """

    #Add self loops to each node
    n = A.shape[0]
    A += sp.identity(n)

    #Calculate degree and inverse of node degress for each node
    degs = A.dot(np.ones(n))
    inv_degs = np.power(degs, -1)

    #Create sparse diagonal matrix
    D = sp.diags(inv_degs)
    
    A_normalized = D.dot(A)

    return A_normalized

#Apply normalization to all graphs
adj = [normalize_adjacency(A) for A in adj]

# Split data into training and test sets
adj_train = list()
features_train = list()
y_train = list()
adj_test = list()
features_test = list()
proteins_test = list()
with open('graph_labels.txt', 'r') as f:
    for i,line in enumerate(f):
        t = line.split(',')
        #Create training data only for graphs that have labels
        if len(t[1][:-1]) > 0:
            adj_train.append(adj[i])
            features_train.append(features[i])
            y_train.append(int(t[1][:-1]))

In [4]:
#Helper function for converting sparse matrices to torch tensors.
def sparse_mx_to_torch_sparse_tensor(sparse_mx):
    """
    Function that converts a Scipy sparse matrix to a sparse Torch tensor
    """
    sparse_mx = sparse_mx.tocoo().astype(np.float32)
    indices = torch.from_numpy(np.vstack((sparse_mx.row, sparse_mx.col)).astype(np.int64))
    values = torch.from_numpy(sparse_mx.data)
    shape = torch.Size(sparse_mx.shape)
    return torch.sparse.FloatTensor(indices, values, shape)

In [5]:
class GNN(nn.Module):
    """
    Message passing model that consists of 2 message passing layers
    and the sum aggregation function
    """
    def __init__(self, input_dim, hidden_dim, dropout, n_class):
        super(GNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim, n_class)
        self.bn = nn.BatchNorm1d(hidden_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x_in, adj, idx):
        # first message passing layer
        x = self.fc1(x_in)
        x = self.relu(torch.mm(adj, x))
        x = self.dropout(x)

        # second message passing layer
        x = self.fc2(x)
        x = self.relu(torch.mm(adj, x))
        
        # sum aggregator
        #reshape and repeat idx to match feature dimensions for aggregation
        idx = idx.unsqueeze(1).repeat(1, x.size(1))
        #create a zero tensor with shape (number_of_graphs, hidden_dim)
        out = torch.zeros(torch.max(idx)+1, x.size(1)).to(x_in.device)
        #sum node features x grouped by graph indices idx
        out = out.scatter_add_(0, idx, x)
        
        # batch normalization layer
        out = self.bn(out)

        # mlp to produce output
        out = self.relu(self.fc3(out))
        out = self.dropout(out)
        out = self.fc4(out)

        return out

Hyperparameters and training loop

In [6]:
# Initialize device
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# Hyperparameters
epochs = 50
batch_size = 64
n_hidden = 64
n_input = 86
dropout = 0.2
learning_rate = 0.001
n_class = 18 #number of protein classes

#Get the number of training graphs available
N_train = len(adj_train)

# Initializes model and optimizer
model = GNN(n_input, n_hidden, dropout, n_class).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
loss_function = nn.CrossEntropyLoss()

# Train model
for epoch in range(epochs):
    t = time.time()
    model.train()
    train_loss = 0
    correct = 0
    count = 0
    # Iterate over the batches
    for i in range(0, N_train, batch_size):
        adj_batch = list()
        features_batch = list()
        idx_batch = list()
        y_batch = list()

        # Create tensors
        for j in range(i, min(N_train, i+batch_size)):
            n = adj_train[j].shape[0]
            adj_batch.append(adj_train[j]+sp.identity(n))
            features_batch.append(features_train[j])
            idx_batch.extend([j-i]*n)
            y_batch.append(y_train[j])

        #Combine all adjacency matrices in the batch
        adj_batch = sp.block_diag(adj_batch)
        features_batch = np.vstack(features_batch)

        #Covert batch data to PyTorch tensors and move to device
        adj_batch = sparse_mx_to_torch_sparse_tensor(adj_batch).to(device)
        features_batch = torch.FloatTensor(features_batch).to(device)
        idx_batch = torch.LongTensor(idx_batch).to(device)
        y_batch = torch.LongTensor(y_batch).to(device)

        #Reset gradients before backpropagation
        optimizer.zero_grad()

        #Forward pass and loss calculation
        output = model(features_batch, adj_batch, idx_batch)
        loss = loss_function(output, y_batch)

        #Accumulate loss
        train_loss += loss.item() * output.size(0)
        count += output.size(0)

        #Get predicted classes
        preds = output.max(1)[1].type_as(y_batch)
        correct += torch.sum(preds.eq(y_batch).double())

        #Backpropagation and optimizer step
        loss.backward()
        optimizer.step()

    if epoch % 5 == 0:
        print('Epoch: {:03d}'.format(epoch+1),
              'loss_train: {:.4f}'.format(train_loss / count),
              'acc_train: {:.4f}'.format(correct / count),
              'time: {:.4f}s'.format(time.time() - t))


/tmp/ipykernel_36000/2843814439.py:10: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:654.)
  return torch.sparse.FloatTensor(indices, values, shape)


Epoch: 001 loss_train: 2.5575 acc_train: 0.2191 time: 12.5561s
Epoch: 006 loss_train: 2.0684 acc_train: 0.3601 time: 12.5986s
Epoch: 011 loss_train: 1.9377 acc_train: 0.4005 time: 12.5923s
Epoch: 016 loss_train: 1.8826 acc_train: 0.4124 time: 12.6078s
Epoch: 021 loss_train: 1.8340 acc_train: 0.4249 time: 12.3391s
Epoch: 026 loss_train: 1.7949 acc_train: 0.4374 time: 12.4332s
Epoch: 031 loss_train: 1.7555 acc_train: 0.4529 time: 12.5981s
Epoch: 036 loss_train: 1.7131 acc_train: 0.4625 time: 12.5210s
Epoch: 041 loss_train: 1.7133 acc_train: 0.4648 time: 12.8225s
Epoch: 046 loss_train: 1.6828 acc_train: 0.4719 time: 12.6479s
